In [1]:
import json
import pandas as pd
import yfinance as yf
from dateutil import parser as dateparser
import numpy as np
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.stattools import grangercausalitytests
from statsmodels.tsa.api import VAR

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
START = "2010-01-01"
END = "2025-07-31"
BASE = "/home/influx/Desktop/FinLLMRL"
STOCKS = ['AAPL', 'BA', 'GS', 'JPM']

WINDOW = 7
HALF_LIFE = 2.5
MX_LAG = 5

In [3]:
def parse_date(item):
    """Parse ISO-like date string in item['date'] into pandas.Timestamp (date only)."""
    return pd.Timestamp(dateparser.parse(str(item['date'])).date())

In [4]:
def label_to_num(label):
    """Map sentiment label to -1/0/+1."""
    l = str(label).strip().lower() if label else ""
    if l == "positive": return 1
    if l == "negative": return -1
    return 0

def signed_score(label, score):
    """
    Convert sentiment_score [0,1] → signed [-1,1].
    - Positive:  score * +1
    - Negative:  score * -1
    - Neutral/other:  0
    """
    if score is None:
        return 0.0
    sgn = label_to_num(label)
    return float(score) * sgn

In [5]:
# ---------- Helper function ----------
def fingpt_signed_score(scores):
    """
    Convert fingpt_score distribution [neutral, positive, negative]
    to a single signed score: neutral=0, positive=+1, negative=-1
    """
    if scores is None:
        return None
    neutral, positive, negative = scores
    return 0*neutral + 1*positive + (-1)*negative

In [6]:
def preparing_news(stock):
    with open(BASE + f"/Data/DOW30_Final/{stock}_final.json", "r") as f:
        news = json.load(f)

    data = {}
    for it in news:
        dt = parse_date(it)
        lab = it.get("sentiment")
        sc  = it.get("sentiment_score", None)
        fg  = it.get("fingpt_score", None)
    
        if dt not in data:
            data[dt] = []
    
        data[dt].append({
            "finbert_score": signed_score(lab, sc),
            "fingpt_score": fingpt_signed_score(fg)
        })

    result = []
    
    for dt, items in data.items():
        finbert_scores = [it["finbert_score"] for it in items if it["finbert_score"] is not None]
        fingpt_scores  = [it["fingpt_score"]  for it in items if it["fingpt_score"]  is not None]
    
        avg_finbert = np.mean(finbert_scores) if finbert_scores else None
        avg_fingpt  = np.mean(fingpt_scores)  if fingpt_scores else None
    
        result.append({
            "date": dt,
            "avg_finbert": avg_finbert,
            "avg_fingpt": avg_fingpt
        })

    df = pd.DataFrame(result).sort_values("date")
    df = df.loc[
        (df['date'] >= pd.to_datetime(START)) &
        (df['date'] <= pd.to_datetime(END))
    ].reset_index(drop=True)

    return df

In [7]:
def preparing_OLCHV(stock):
    ST = pd.to_datetime(START)
    ED   = pd.to_datetime(END)
    
    prices = yf.download(
        stock,
        start=ST.strftime("%Y-%m-%d"),
        end=(ED + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
        progress=False,
        auto_adjust=True
    ).reset_index()

    prices.columns = ["_".join([str(c) for c in col if c]).strip() for col in prices.columns.values]

    prices = prices.rename(columns={
        f"Close_{stock}": "Close",
        f"Volume_{stock}": "Volume",
        "Date": "date"
    })

    prices = prices.drop(columns=[f'High_{stock}', f'Low_{stock}', f'Open_{stock}'])
    return prices

In [8]:
def merge_data(prices, df):
    prices = prices.copy()
    df = df.copy()
    
    prices['date'] = pd.to_datetime(prices['date'])
    df['date'] = pd.to_datetime(df['date'])
    
    prices = prices.sort_values('date').drop_duplicates('date', keep='last')
    df = df.sort_values('date').drop_duplicates('date', keep='last')
    
    start = min(prices['date'].min(), df['date'].min())
    end   = max(prices['date'].max(), df['date'].max())
    all_days = pd.date_range(start, end, freq='D', name='date')  
    
    prices_full = (
        prices.set_index('date')
              .reindex(all_days)      
              .ffill()                
    )
    
    df_full = (
        df.set_index('date')
          .reindex(all_days, fill_value=0)
    )
    
    out = prices_full.join(df_full, how='outer').reset_index().rename(columns={'index':'date'})
    out = out.sort_values("date").reset_index(drop=True)
    out["return"] = np.log(out["Close"] / out["Close"].shift(1))
    out = out.dropna(subset=["return"]).reset_index(drop=True)

    out = out.sort_values('date').reset_index(drop=True)
    
    out['finbert7'] = past_exp_weighted_avg(out['avg_finbert'], window=WINDOW, half_life=HALF_LIFE, exclude_today=False)
    out['fingpt7']  = past_exp_weighted_avg(out['avg_fingpt'],  window=WINDOW, half_life=HALF_LIFE, exclude_today=False)

    return out

In [9]:
def past_exp_weighted_avg(series: pd.Series, window=7, half_life=1.5, exclude_today=False):
    """
    Exponentially weighted average over the past `window` days.
    - exclude_today=True -> uses t-1..t-window
    - Row-wise renormalization handles missing values at the start.
    """
    lags = np.arange(1, window + 1) if exclude_today else np.arange(0, window)
    lam = np.log(2) / half_life
    w = np.exp(-lam * lags)  

    lagged = pd.concat([series.shift(l) for l in lags], axis=1)

    weighted = lagged.mul(w, axis=1)
    weight_sums = (~lagged.isna()).mul(w, axis=1).sum(axis=1)
    out = weighted.sum(axis=1) / weight_sums
    out[weight_sums == 0] = np.nan
    return out

In [10]:
def adf_test(series, title=""):
    print(f"--- ADF Test: {title} ---")
    result = adfuller(series.dropna(), autolag='AIC')
    labels = ['ADF Statistic', 'p-value', '# Lags Used', '# Observations Used']
    out = dict(zip(labels, result[0:4]))
    for key, val in out.items():
        print(f"{key} : {val}")
    for key, val in result[4].items():
        print(f"Critical Value ({key}) : {val}")
    if result[1] <= 0.05:
        print("✅ Reject H0 → Stationary")
    else:
        print("❌ Fail to Reject H0 → Non-stationary")
    print("\n")

In [11]:
def granger_test(df, xvar, yvar="return", max_lag=5):
    print(f"\n=== Granger causality test: Does {xvar} → {yvar}? ===")
    test_result = grangercausalitytests(df[[yvar, xvar]], max_lag, verbose=True)
    return test_result

In [12]:
prices = preparing_OLCHV("AAPL")
news = preparing_news("AAPL")

df = merge_data(prices, news)

adf_test(df["Close"], title="Close Price")

adf_test(df["return"], title="Log Return")

--- ADF Test: Close Price ---
ADF Statistic : 0.17489979981356057
p-value : 0.97081813633798
# Lags Used : 33
# Observations Used : 5653
Critical Value (1%) : -3.4315073097262423
Critical Value (5%) : -2.862051418757438
Critical Value (10%) : -2.567042226588413
❌ Fail to Reject H0 → Non-stationary


--- ADF Test: Log Return ---
ADF Statistic : -16.526235290311927
p-value : 2.0428884821297832e-29
# Lags Used : 19
# Observations Used : 5667
Critical Value (1%) : -3.4315044493620004
Critical Value (5%) : -2.8620501549989936
Critical Value (10%) : -2.5670415538515483
✅ Reject H0 → Stationary




In [13]:
# granger_test(df, "avg_finbert", "return", MX_LAG)
# granger_test(df, "finbert7", "return", MX_LAG)
# granger_test(df, "avg_fingpt", "return", MX_LAG)
# granger_test(df, "fingpt7", "return", MX_LAG)

In [14]:
data = df[['return', 'finbert7', 'fingpt7']].dropna()

sel = VAR(data).select_order(MX_LAG)

lag_candidates = [sel.aic, sel.bic, sel.hqic, sel.fpe]
lag = next((int(x) for x in lag_candidates if x is not None and x > 0), 5)

var_res = VAR(data).fit(lag)

print(f"Selected VAR lag order: {lag}")
# print(var_res.summary())

Selected VAR lag order: 5


In [15]:
for s in ['finbert7', 'fingpt7']:
    test = var_res.test_causality('return', [s], kind='f')
    print(f"\nGranger test: {s} → ret")
    print(f"  F-stat = {test.test_statistic:.3f}")
    print(f"  p-value = {test.pvalue:.4g}")

test_joint = var_res.test_causality('return', ['finbert7', 'fingpt7'], kind='f')
print(test_joint.summary())


Granger test: finbert7 → ret
  F-stat = 1.245
  p-value = 0.2852

Granger test: fingpt7 → ret
  F-stat = 4.391
  p-value = 0.0005364
Granger causality F-test. H_0: ['finbert7', 'fingpt7'] do not Granger-cause return. Conclusion: reject H_0 at 5% significance level.
Test statistic Critical value p-value      df    
-------------------------------------------------
         3.914          1.831   0.000 (10, 16998)
-------------------------------------------------


In [16]:
prices = preparing_OLCHV("BA")
news = preparing_news("BA")

df = merge_data(prices, news)

adf_test(df["Close"], title="Close Price")
adf_test(df["return"], title="Log Return")

--- ADF Test: Close Price ---
ADF Statistic : -1.858701209946665
p-value : 0.3517316419268725
# Lags Used : 33
# Observations Used : 5653
Critical Value (1%) : -3.4315073097262423
Critical Value (5%) : -2.862051418757438
Critical Value (10%) : -2.567042226588413
❌ Fail to Reject H0 → Non-stationary


--- ADF Test: Log Return ---
ADF Statistic : -13.511272209080497
p-value : 2.850205418491391e-25
# Lags Used : 33
# Observations Used : 5653
Critical Value (1%) : -3.4315073097262423
Critical Value (5%) : -2.862051418757438
Critical Value (10%) : -2.567042226588413
✅ Reject H0 → Stationary




In [17]:
# granger_test(df, "avg_finbert", "return", MX_LAG)
# granger_test(df, "finbert7", "return", MX_LAG)
# granger_test(df, "avg_fingpt", "return", MX_LAG)
# granger_test(df, "fingpt7", "return", MX_LAG)

In [18]:
data = df[['return', 'finbert7', 'fingpt7']].dropna()

sel = VAR(data).select_order(MX_LAG)

lag_candidates = [sel.aic, sel.bic, sel.hqic, sel.fpe]
lag = next((int(x) for x in lag_candidates if x is not None and x > 0), 5)

var_res = VAR(data).fit(lag)

print(f"Selected VAR lag order: {lag}")
# print(var_res.summary())

Selected VAR lag order: 2


In [19]:
for s in ['finbert7', 'fingpt7']:
    test = var_res.test_causality('return', [s], kind='f')
    print(f"\nGranger test: {s} → ret")
    print(f"  F-stat = {test.test_statistic:.3f}")
    print(f"  p-value = {test.pvalue:.4g}")

test_joint = var_res.test_causality('return', ['finbert7', 'fingpt7'], kind='f')
print(test_joint.summary())


Granger test: finbert7 → ret
  F-stat = 0.060
  p-value = 0.9413

Granger test: fingpt7 → ret
  F-stat = 0.176
  p-value = 0.8386
Granger causality F-test. H_0: ['finbert7', 'fingpt7'] do not Granger-cause return. Conclusion: fail to reject H_0 at 5% significance level.
Test statistic Critical value p-value     df    
------------------------------------------------
        0.1765          2.372   0.951 (4, 17034)
------------------------------------------------


In [20]:
prices = preparing_OLCHV("GS")
news = preparing_news("GS")

df = merge_data(prices, news)

adf_test(df["Close"], title="Close Price")
adf_test(df["return"], title="Log Return")

--- ADF Test: Close Price ---
ADF Statistic : 2.0232799504030234
p-value : 0.9987024479195521
# Lags Used : 29
# Observations Used : 5657
Critical Value (1%) : -3.4315064910339945
Critical Value (5%) : -2.862051057045153
Critical Value (10%) : -2.567042034037996
❌ Fail to Reject H0 → Non-stationary


--- ADF Test: Log Return ---
ADF Statistic : -21.790842088795593
p-value : 0.0
# Lags Used : 10
# Observations Used : 5676
Critical Value (1%) : -3.431502618010906
Critical Value (5%) : -2.8620493458757355
Critical Value (10%) : -2.567041123130861
✅ Reject H0 → Stationary




In [21]:
# granger_test(df, "avg_finbert", "return", MX_LAG)
# granger_test(df, "finbert7", "return", MX_LAG)
# granger_test(df, "avg_fingpt", "return", MX_LAG)
# granger_test(df, "fingpt7", "return", MX_LAG)

In [22]:
data = df[['return', 'finbert7', 'fingpt7']].dropna()

sel = VAR(data).select_order(MX_LAG)

lag_candidates = [sel.aic, sel.bic, sel.hqic, sel.fpe]
lag = next((int(x) for x in lag_candidates if x is not None and x > 0), 5)

var_res = VAR(data).fit(lag)

print(f"Selected VAR lag order: {lag}")
# print(var_res.summary())

Selected VAR lag order: 2


In [23]:
for s in ['finbert7', 'fingpt7']:
    test = var_res.test_causality('return', [s], kind='f')
    print(f"\nGranger test: {s} → ret")
    print(f"  F-stat = {test.test_statistic:.3f}")
    print(f"  p-value = {test.pvalue:.4g}")

test_joint = var_res.test_causality('return', ['finbert7', 'fingpt7'], kind='f')
print(test_joint.summary())


Granger test: finbert7 → ret
  F-stat = 0.257
  p-value = 0.7733

Granger test: fingpt7 → ret
  F-stat = 0.945
  p-value = 0.3885
Granger causality F-test. H_0: ['finbert7', 'fingpt7'] do not Granger-cause return. Conclusion: fail to reject H_0 at 5% significance level.
Test statistic Critical value p-value     df    
------------------------------------------------
         1.047          2.372   0.381 (4, 17034)
------------------------------------------------


In [24]:
prices = preparing_OLCHV("JPM")
news = preparing_news("JPM")

df = merge_data(prices, news)

adf_test(df["Close"], title="Close Price")
adf_test(df["return"], title="Log Return")

--- ADF Test: Close Price ---
ADF Statistic : 2.5737362958511483
p-value : 0.9990698336999587
# Lags Used : 29
# Observations Used : 5657
Critical Value (1%) : -3.4315064910339945
Critical Value (5%) : -2.862051057045153
Critical Value (10%) : -2.567042034037996
❌ Fail to Reject H0 → Non-stationary


--- ADF Test: Log Return ---
ADF Statistic : -17.49693971301723
p-value : 4.406287940526535e-30
# Lags Used : 20
# Observations Used : 5666
Critical Value (1%) : -3.431504653204749
Critical Value (5%) : -2.8620502450602894
Critical Value (10%) : -2.567041601793895
✅ Reject H0 → Stationary




In [25]:
# granger_test(df, "avg_finbert", "return", MX_LAG)
# granger_test(df, "finbert7", "return", MX_LAG)
# granger_test(df, "avg_fingpt", "return", MX_LAG)
# granger_test(df, "fingpt7", "return", MX_LAG)

In [26]:
data = df[['return', 'finbert7', 'fingpt7']].dropna()

sel = VAR(data).select_order(MX_LAG)

lag_candidates = [sel.aic, sel.bic, sel.hqic, sel.fpe]
lag = next((int(x) for x in lag_candidates if x is not None and x > 0), 5)

var_res = VAR(data).fit(lag)

print(f"Selected VAR lag order: {lag}")
# print(var_res.summary())

Selected VAR lag order: 2


In [27]:
for s in ['finbert7', 'fingpt7']:
    test = var_res.test_causality('return', [s], kind='f')
    print(f"\nGranger test: {s} → ret")
    print(f"  F-stat = {test.test_statistic:.3f}")
    print(f"  p-value = {test.pvalue:.4g}")

test_joint = var_res.test_causality('return', ['finbert7', 'fingpt7'], kind='f')
print(test_joint.summary())


Granger test: finbert7 → ret
  F-stat = 0.341
  p-value = 0.7107

Granger test: fingpt7 → ret
  F-stat = 0.294
  p-value = 0.745
Granger causality F-test. H_0: ['finbert7', 'fingpt7'] do not Granger-cause return. Conclusion: fail to reject H_0 at 5% significance level.
Test statistic Critical value p-value     df    
------------------------------------------------
        0.2762          2.372   0.894 (4, 17034)
------------------------------------------------
